# Feature Engineering

##      Walking And Jogging

In [2]:
import matplotlib.pyplot as plt
import iml_assignment as iml
import numpy as np
import pandas as pd

train_df = iml.load_data("train")
test_df = iml.load_data("test")

In [5]:
scaler = iml.ChannelScaler(method="robust").fit(train_df)
X_train, y_train = iml.extract_features_dataset(train_df,impute_method="cubic_spline",scaler=scaler)
X_test, _ = iml.extract_features_dataset(test_df,impute_method="cubic_spline",scaler=scaler)

#baseline feature for general model

Extracting features for 679 samples...
  - Imputing corrupted readings using cubic_spline...
  - Normalizing channels using fitted robust scaler...
Feature extraction complete! Shape: (679, 357)
Extracting features for 403 samples...
  - Imputing corrupted readings using cubic_spline...
  - Normalizing channels using fitted robust scaler...
Feature extraction complete! Shape: (403, 357)


In [6]:
#specialist model feature

clean_train = iml.clean_dataset(train_df,method = "cubic_spline")
clean_test = iml.clean_dataset(test_df,method="cubic_spline")

X_train_eng = iml.extract_all_engineered_features(clean_train)
X_test_eng = iml.extract_all_engineered_features(clean_test)


In [7]:
X_tr_base = X_train.to_numpy()
X_te_base = X_test.to_numpy()
X_tr_eng  = X_train_eng.to_numpy()
X_te_eng  = X_test_eng.to_numpy()
y_tr = y_train.to_numpy()

In [9]:
display(X_tr_base)
display(X_tr_eng)

array([[-4.59988300e-02,  2.67607976e+00, -1.40714094e+01, ...,
         5.71027212e+01,  1.63154421e-01,  1.14210390e+00],
       [ 2.10111208e-01,  2.77638251e+00, -9.87652136e+00, ...,
         3.53778075e+01,  7.98332967e-01,  5.36786951e+00],
       [-3.70180481e-01,  1.88671151e+00, -1.08983417e+01, ...,
         3.60047634e+01,  7.60056184e-03,  2.10396406e-01],
       ...,
       [-1.02287587e-01,  2.57201741e+00, -1.00071615e+01, ...,
         4.56592684e+01,  2.43816696e-03,  5.82088059e-01],
       [ 6.92261615e-02,  2.61504260e+00, -9.93041121e+00, ...,
         4.37368039e+01,  1.45256751e+00,  7.11696309e+00],
       [-2.53807077e-01,  2.20171243e+00, -9.82321915e+00, ...,
         4.70238269e+01,  7.29493928e-01,  6.25046966e+00]],
      shape=(679, 357))

array([[1.34861423e+03, 2.99693924e+03, 1.83619831e+00, ...,
        9.24163065e+00, 2.35694901e+00, 7.97542628e+00],
       [1.68691435e+03, 3.42514358e+03, 1.22058362e+00, ...,
        9.09289460e+00, 2.50738431e+00, 8.01049679e+00],
       [1.23244172e+03, 3.10423563e+03, 1.38940531e+00, ...,
        9.88905127e+00, 2.25983982e+00, 7.32051218e+00],
       ...,
       [1.28404518e+03, 3.07913601e+03, 2.24036967e+00, ...,
        1.03639284e+01, 2.21930493e+00, 6.52994876e+00],
       [1.41696887e+03, 3.23221566e+03, 6.85479050e-01, ...,
        8.50256700e+00, 2.54145582e+00, 9.97552670e+00],
       [1.19577199e+03, 2.82305480e+03, 7.71998679e-01, ...,
        1.09102454e+01, 2.10795217e+00, 8.25616916e+00]], shape=(679, 26))

Engineered Subset has 26 features

In [ ]:
#training classifier Model
from sklearn.ensemble import RandomForestClassifier

base_model = RandomForestClassifier(n_estimators=100, class_weight="balanced", max_depth=8, random_state=42)
base_model.fit(X_tr_base, y_tr)

base_test_preds = base_model.predict(X_te_base)


print("--- Stage 1 Baseline Score ---")
iml.evaluate_against_benchmark(base_test_preds)

--- Stage 1 Baseline Score ---

  Test Benchmark Evaluation (SOTA Pseudo-Ground Truth)
  Benchmark Reference: benchmark_submission.csv (403 samples)
  Matches:     390 / 403 (96.77%)
  Mismatches:  13 / 403 (3.23%)
  Macro F1:    0.9688
  Weighted F1: 0.9673
-------------------------------------------------------
  Top Disagreement Pairs (Predicted -> Benchmark):
    - 'jogging' predicted as 'running': 10 sample(s)
    - 'running' predicted as 'jogging': 2 sample(s)
    - 'climbing_stairs' predicted as 'knees_bending': 1 sample(s)



{'accuracy': 0.9677,
 'accuracy_pct': 96.77,
 'macro_f1': 0.9688,
 'weighted_f1': 0.9673,
 'matches': 390,
 'mismatches': 13,
 'mismatches_df':           id        predicted      benchmark
 0   TE000034          jogging        running
 1   TE000042          jogging        running
 2   TE000072          running        jogging
 3   TE000082          jogging        running
 4   TE000122          running        jogging
 5   TE000133          jogging        running
 6   TE000191          jogging        running
 7   TE000260          jogging        running
 8   TE000279          jogging        running
 9   TE000284          jogging        running
 10  TE000308  climbing_stairs  knees_bending
 11  TE000325          jogging        running
 12  TE000355          jogging        running,
 'confusion_pairs': predicted        benchmark    
 jogging          running          10
 running          jogging           2
 climbing_stairs  knees_bending     1
 dtype: int64,
 'report_df':                  p

If this model predicts Running or Jogging, we can route that prediction to the specialist model

In [16]:
TARGET_CLASS = ["running","jogging"]

special_train_mask = np.isin(y_tr,TARGET_CLASS)

X_tr_spc = X_tr_eng[special_train_mask]
y_tr_spc = y_tr[special_train_mask]

print(f"Training specialist model on {len(y_tr_spc)} samples for classes: {TARGET_CLASS}")

Training specialist model on 127 samples for classes: ['running', 'jogging']


In [19]:
DEPTH = [1,2,3,5,8,10,None]

for n in DEPTH:
    specialist_model = RandomForestClassifier(n_estimators=100, random_state=42,max_depth=n)
    specialist_model.fit(X_tr_spc,y_tr_spc)
    